# Spatial Arrangement Method (SpAM) Analysis
## Real-Coordinate Pipeline with Cross-Task Link to VFT

This notebook completes the SpAM phase using real `droppedwords` coordinates from `responses.json`.

### Covered in this notebook
- Real SpAM coordinate extraction (no simulation)
- Subject-level and consensus semantic distance matrices
- MDS maps and hierarchical clustering
- RQ3: SpAM distance vs VFT transition latency (Spearman)
- BH-FDR correction across domains
- Cross-task summary plots and tables


In [1]:
import pandas as _pd

_ORIG_READ_CSV = _pd.read_csv

def _read_csv_hindi_only(*args, **kwargs):
    df = _ORIG_READ_CSV(*args, **kwargs)
    path = ""
    if args:
        path = str(args[0])
    elif "filepath_or_buffer" in kwargs:
        path = str(kwargs.get("filepath_or_buffer", ""))

    if path.replace("\\", "/").endswith("vft_responses.csv") and "language_type" in df.columns:
        df = df[df["language_type"].astype(str).str.strip().str.lower().eq("hindi/hinglish")].copy()
    return df

_pd.read_csv = _read_csv_hindi_only
print("Notebook guard active: vft_responses.csv will be Hindi/Hinglish-only")

Notebook guard active: vft_responses.csv will be Hindi/Hinglish-only


In [2]:
import os
import sys
import json
import importlib
import subprocess
from itertools import combinations

# ------------------------------------------------------------
# 0) Environment-safe imports
# ------------------------------------------------------------
AUTO_INSTALL_CORE = True


def ensure_import(module_name, pip_name=None):
    try:
        return importlib.import_module(module_name)
    except ImportError:
        if AUTO_INSTALL_CORE and pip_name:
            print(f"Installing missing package: {pip_name}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
            return importlib.import_module(module_name)
        raise


np = ensure_import("numpy", "numpy")
pd = ensure_import("pandas", "pandas")
matplotlib = ensure_import("matplotlib", "matplotlib")
matplotlib.use("Agg")
plt = ensure_import("matplotlib.pyplot")
sns = ensure_import("seaborn", "seaborn")

stats = ensure_import("scipy.stats", "scipy")
spatial_distance = ensure_import("scipy.spatial.distance", "scipy")
pdist = spatial_distance.pdist
squareform = spatial_distance.squareform
cluster_h = ensure_import("scipy.cluster.hierarchy", "scipy")
linkage = cluster_h.linkage
fcluster = cluster_h.fcluster

sk_manifold = ensure_import("sklearn.manifold", "scikit-learn")
MDS = sk_manifold.MDS
sk_metrics = ensure_import("sklearn.metrics", "scikit-learn")
silhouette_score = sk_metrics.silhouette_score

smf = ensure_import("statsmodels.formula.api", "statsmodels")
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------
# 1) Config and helpers
# ------------------------------------------------------------
IMGDIR = "images"
os.makedirs(IMGDIR, exist_ok=True)
DOMAINS = ["animals", "foods", "colours", "body-parts"]
DOMAIN_COLORS = {
    "animals": "#4E79A7",
    "foods": "#F28E2B",
    "colours": "#E15759",
    "body-parts": "#76B7B2",
}

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "figure.dpi": 140,
        "savefig.dpi": 160,
        "savefig.bbox": "tight",
    }
)


def clean_word(text):
    if pd.isna(text):
        return ""
    return str(text).strip().lower()


def parse_float(value):
    try:
        if value in [None, "", "nan"]:
            return np.nan
        return float(value)
    except Exception:
        return np.nan


def pair_key(w1, w2):
    a, b = sorted([clean_word(w1), clean_word(w2)])
    return f"{a}||{b}"


# ------------------------------------------------------------
# 2) Load source data
# ------------------------------------------------------------
vft = pd.read_csv("vft_responses.csv", encoding="utf-8")
vft["subject_id"] = vft["subject_id"].astype(str)
vft["domain"] = vft["domain"].astype(str).str.strip().str.lower()
vft["word"] = vft["word"].apply(clean_word)
vft["position"] = pd.to_numeric(vft["position"], errors="coerce")
vft["rt_ms"] = pd.to_numeric(vft["rt_ms"], errors="coerce")
vft = vft[vft["domain"].isin(DOMAINS)].copy()

with open("responses.json", encoding="utf-8") as f:
    raw_json = json.load(f)
sessions = raw_json.get("fluency-spam", raw_json)

# ------------------------------------------------------------
# 3) Extract real SpAM coordinates from droppedwords
# ------------------------------------------------------------
spam_rows = []
for session_id, session_payload in sessions.items():
    if not isinstance(session_payload, dict):
        continue
    sid = str(session_payload.get("subject_id", session_id))
    trials = session_payload.get("data", [])
    if not isinstance(trials, list):
        continue

    for trial in trials:
        if not isinstance(trial, dict):
            continue

        domain = str(trial.get("domain", "")).strip().lower()
        dropped = trial.get("droppedwords")
        if domain not in DOMAINS or not isinstance(dropped, list):
            continue

        for move_index, item in enumerate(dropped):
            if not isinstance(item, dict):
                continue

            w = clean_word(item.get("word", ""))
            if not w:
                continue

            x_norm = item.get("x_norm", np.nan)
            y_norm = item.get("y_norm", np.nan)
            if (x_norm is None or pd.isna(x_norm)) and item.get("x_px") is not None and item.get("canvas_width"):
                x_norm = float(item["x_px"]) / max(float(item["canvas_width"]), 1.0)
            if (y_norm is None or pd.isna(y_norm)) and item.get("y_px") is not None and item.get("canvas_height"):
                y_norm = float(item["y_px"]) / max(float(item["canvas_height"]), 1.0)

            spam_rows.append(
                {
                    "subject_id": sid,
                    "session_id": str(session_id),
                    "domain": domain,
                    "word": w,
                    "x_norm": parse_float(x_norm),
                    "y_norm": parse_float(y_norm),
                    "move_index": move_index,
                }
            )

spam = pd.DataFrame(spam_rows)
spam = spam.dropna(subset=["x_norm", "y_norm"])

# Keep final placement per subject-domain-word (last move event)
spam = (
    spam.sort_values(["subject_id", "domain", "word", "move_index"])
    .groupby(["subject_id", "domain", "word"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

print("SpAM extraction complete")
print(f"SpAM coordinate rows: {len(spam)}")
print(f"Participants with SpAM: {spam['subject_id'].nunique()}")
print("Domain coverage (subject-domain pairs):")
print(spam[["subject_id", "domain"]].drop_duplicates()["domain"].value_counts())

# ------------------------------------------------------------
# 4) Subject-level pairwise distances + consensus matrices
# ------------------------------------------------------------
pair_rows = []
for (sid, domain), grp in spam.groupby(["subject_id", "domain"]):
    grp = grp[["word", "x_norm", "y_norm"]].dropna().drop_duplicates(subset=["word"])
    if len(grp) < 3:
        continue

    words = grp["word"].tolist()
    coords = grp[["x_norm", "y_norm"]].values
    dist = squareform(pdist(coords, metric="euclidean"))

    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            pair_rows.append(
                {
                    "subject_id": sid,
                    "domain": domain,
                    "word_1": words[i],
                    "word_2": words[j],
                    "pair_key": pair_key(words[i], words[j]),
                    "spam_distance": float(dist[i, j]),
                }
            )

pair_df = pd.DataFrame(pair_rows)

consensus = {}
consensus_pair_rows = []
for domain in DOMAINS:
    dom_pairs = pair_df[pair_df["domain"] == domain].copy()
    if len(dom_pairs) == 0:
        continue

    agg = (
        dom_pairs.groupby(["domain", "pair_key", "word_1", "word_2"], as_index=False)
        .agg(
            spam_distance_mean=("spam_distance", "mean"),
            n_subjects=("subject_id", "nunique"),
        )
    )

    # Use pairs supported by at least 5 participants as planned.
    agg = agg[agg["n_subjects"] >= 5].copy()
    if len(agg) < 3:
        continue

    consensus_pair_rows.append(agg)

    words = sorted(set(agg["word_1"]).union(set(agg["word_2"])))
    idx = {w: i for i, w in enumerate(words)}
    mat = np.full((len(words), len(words)), np.nan, dtype=float)
    np.fill_diagonal(mat, 0.0)

    for _, row in agg.iterrows():
        i = idx[row["word_1"]]
        j = idx[row["word_2"]]
        d = float(row["spam_distance_mean"])
        mat[i, j] = d
        mat[j, i] = d

    # Fill any sparse missing pairs with domain median distance.
    med = np.nanmedian(mat)
    mat = np.where(np.isnan(mat), med, mat)

    consensus[domain] = {"words": words, "distance": mat, "pairs": agg}

consensus_pairs_df = pd.concat(consensus_pair_rows, ignore_index=True) if len(consensus_pair_rows) > 0 else pd.DataFrame()

print("Consensus distance matrices ready")
for d in consensus:
    print(f"  {d}: {len(consensus[d]['words'])} words, {len(consensus[d]['pairs'])} supported pairs")

# ------------------------------------------------------------
# 5) MDS maps and hierarchical clustering (Ward)
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
cluster_rows = []

for ax, domain in zip(axes.flatten(), DOMAINS):
    if domain not in consensus:
        ax.axis("off")
        continue

    words = consensus[domain]["words"]
    dist_mat = consensus[domain]["distance"]

    if len(words) < 3:
        ax.axis("off")
        continue

    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42, max_iter=1200, normalized_stress=False)
    coords = mds.fit_transform(dist_mat)

    # Cluster selection by silhouette on precomputed distance.
    max_k = min(6, len(words) - 1)
    best_k = 2
    best_s = -1.0
    Z = linkage(squareform(dist_mat, checks=False), method="ward")

    for k in range(2, max_k + 1):
        labels = fcluster(Z, k, criterion="maxclust")
        if len(np.unique(labels)) < 2:
            continue
        try:
            sil = silhouette_score(dist_mat, labels, metric="precomputed")
            if sil > best_s:
                best_s = sil
                best_k = k
        except Exception:
            pass

    labels = fcluster(Z, best_k, criterion="maxclust")
    cluster_rows.append({"domain": domain, "k": best_k, "silhouette": best_s, "n_words": len(words)})

    cmap = plt.cm.get_cmap("tab10", best_k)
    for i, w in enumerate(words):
        ax.scatter(coords[i, 0], coords[i, 1], s=65, color=cmap(labels[i] - 1), edgecolors="white", alpha=0.88)
        ax.text(coords[i, 0] + 0.004, coords[i, 1] + 0.004, w, fontsize=7)

    ax.set_title(f"{domain.capitalize()} | MDS + Ward clusters (k={best_k})")
    ax.set_xlabel("MDS dim 1")
    ax.set_ylabel("MDS dim 2")

fig.suptitle("SpAM Consensus Semantic Maps (Real Coordinates)", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(IMGDIR, "spam_fig01_mds_consensus_maps.png"))
plt.close()

cluster_df = pd.DataFrame(cluster_rows)

# ------------------------------------------------------------
# 6) RQ3: SpAM distance vs VFT transition latency
# ------------------------------------------------------------
# Transition latency: IRT for current word in a consecutive pair.
trans_rows = []
for (sid, domain), grp in vft.groupby(["subject_id", "domain"]):
    grp = grp.sort_values("position")
    if len(grp) < 2:
        continue
    rows = grp[["word", "rt_ms", "position"]].dropna().values.tolist()
    for i in range(1, len(rows)):
        prev_word = clean_word(rows[i - 1][0])
        curr_word = clean_word(rows[i][0])
        curr_irt = float(rows[i][1])
        prev_irt = float(rows[i - 1][1])
        trans_rows.append(
            {
                "subject_id": sid,
                "domain": domain,
                "prev_word": prev_word,
                "curr_word": curr_word,
                "pair_key": pair_key(prev_word, curr_word),
                "irt_transition_ms": curr_irt,
                "irt_gap_ms": abs(curr_irt - prev_irt),
            }
        )

trans_df = pd.DataFrame(trans_rows)

# Subject-level merge first, then domain correlations.
subject_pair_spam = pair_df[["subject_id", "domain", "pair_key", "spam_distance"]].copy()
rq3_df = trans_df.merge(subject_pair_spam, on=["subject_id", "domain", "pair_key"], how="inner")

print("\nRQ3 merged rows (transition pairs with SpAM distance):", len(rq3_df))

rq3_rows = []
for domain in DOMAINS:
    sub = rq3_df[rq3_df["domain"] == domain].dropna(subset=["spam_distance", "irt_transition_ms"])
    if len(sub) < 10:
        continue
    rho, pval = stats.spearmanr(sub["spam_distance"], sub["irt_transition_ms"])
    rq3_rows.append(
        {
            "domain": domain,
            "n_pairs": len(sub),
            "spearman_rho": float(rho),
            "p_raw": float(pval),
        }
    )

rq3_res = pd.DataFrame(rq3_rows)
if len(rq3_res) > 0:
    _, p_bh, _, _ = multipletests(rq3_res["p_raw"], method="fdr_bh")
    rq3_res["p_bh"] = p_bh
    rq3_res["significant_bh"] = rq3_res["p_bh"] < 0.05

print("\nRQ3 domain-wise Spearman results (SpAM distance vs VFT transition IRT):")
print(rq3_res.to_string(index=False) if len(rq3_res) > 0 else "No domain had enough merged pairs.")

# Pooled regression summary for effect direction.
if len(rq3_df) >= 30:
    pooled_model = smf.ols("irt_transition_ms ~ spam_distance + C(domain)", data=rq3_df).fit()
    print("\nPooled OLS (descriptive):")
    print(
        f"spam_distance beta={pooled_model.params.get('spam_distance', np.nan):.2f}, "
        f"p={pooled_model.pvalues.get('spam_distance', np.nan):.4f}, R^2={pooled_model.rsquared:.3f}"
    )

# ------------------------------------------------------------
# 7) RQ3 figures and summary tables
# ------------------------------------------------------------
if len(rq3_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Scatter (all domains combined)
    ax = axes[0]
    for d in DOMAINS:
        sub = rq3_df[rq3_df["domain"] == d]
        if len(sub) == 0:
            continue
        ax.scatter(
            sub["spam_distance"],
            sub["irt_transition_ms"],
            s=18,
            alpha=0.35,
            color=DOMAIN_COLORS[d],
            label=d.capitalize(),
        )
    ax.set_xlabel("SpAM distance (subject-level)")
    ax.set_ylabel("VFT transition IRT (ms)")
    ax.set_title("RQ3 scatter: SpAM distance vs transition latency")
    ax.legend(fontsize=8)

    # Forest/bar summary by domain
    ax2 = axes[1]
    if len(rq3_res) > 0:
        order = rq3_res.sort_values("spearman_rho")
        bars = ax2.barh(
            [d.capitalize() for d in order["domain"]],
            order["spearman_rho"],
            color=[DOMAIN_COLORS[d] for d in order["domain"]],
            alpha=0.85,
        )
        ax2.axvline(0, color="black", lw=0.8)
        ax2.set_xlabel("Spearman rho")
        ax2.set_title("RQ3 by domain (BH corrected)")
        for bar, pbh in zip(bars, order["p_bh"]):
            mark = "*" if pbh < 0.05 else "ns"
            ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, mark, va="center")
    else:
        ax2.text(0.5, 0.5, "No domain-level result", ha="center", va="center")
        ax2.axis("off")

    fig.suptitle("RQ3: Cross-task Convergence (SpAM -> VFT)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(IMGDIR, "spam_fig09_rq3_correlation.png"))
    plt.close()

# Dedicated summary figure requested in plan
if len(rq3_res) > 0:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    order = rq3_res.sort_values("spearman_rho")
    bars = ax.barh(
        [d.capitalize() for d in order["domain"]],
        order["spearman_rho"],
        color=[DOMAIN_COLORS[d] for d in order["domain"]],
        alpha=0.85,
    )
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Spearman rho")
    ax.set_title("Cross-task summary (domain-level RQ3)")
    for bar, n, pbh in zip(bars, order["n_pairs"], order["p_bh"]):
        mark = "*" if pbh < 0.05 else "ns"
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, f"n={n}, {mark}", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(IMGDIR, "spam_fig10_cross_task_summary.png"))
    plt.close()

# ------------------------------------------------------------
# 8) Save analysis tables
# ------------------------------------------------------------
cluster_df.to_csv("table_spam_clusters.csv", index=False)
if len(rq3_res) > 0:
    rq3_res.to_csv("table_spam_rq3_results.csv", index=False)
if len(consensus_pairs_df) > 0:
    consensus_pairs_df.to_csv("table_spam_consensus_pairs.csv", index=False)

print("\nSaved figures:")
print("  spam_fig01_mds_consensus_maps.png")
print("  spam_fig09_rq3_correlation.png")
print("  spam_fig10_cross_task_summary.png")
print("Saved tables:")
print("  table_spam_clusters.csv")
print("  table_spam_rq3_results.csv")
print("  table_spam_consensus_pairs.csv")

print("\nSpAM notebook complete.")

SpAM extraction complete
SpAM coordinate rows: 1033
Participants with SpAM: 35
Domain coverage (subject-domain pairs):
domain
animals       35
foods         35
body-parts    24
colours       11
Name: count, dtype: int64
Consensus distance matrices ready
  animals: 21 words, 47 supported pairs
  foods: 3 words, 3 supported pairs
  colours: 10 words, 40 supported pairs
  body-parts: 5 words, 10 supported pairs


c:\Users\kotad\OneDrive\Desktop\BRSM new\Hindi-Fluency\.venv\Lib\site-packages\sklearn\manifold\_mds.py:744: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9. To suppress this warning, provide some value of `n_init`.
  warnings.warn(
c:\Users\kotad\OneDrive\Desktop\BRSM new\Hindi-Fluency\.venv\Lib\site-packages\sklearn\manifold\_mds.py:754: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
c:\Users\kotad\OneDrive\Desktop\BRSM new\Hindi-Fluency\.venv\Lib\site-packages\sklearn\manifold\_mds.py:771: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(
C:\Users\kotad\AppData\Local\Temp\ipykernel_29156\3414019259.py:290: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``m


RQ3 merged rows (transition pairs with SpAM distance): 565

RQ3 domain-wise Spearman results (SpAM distance vs VFT transition IRT):
    domain  n_pairs  spearman_rho    p_raw     p_bh  significant_bh
   animals      186      0.179033 0.014485 0.057939           False
     foods      205      0.092664 0.186341 0.186341           False
   colours       36      0.335393 0.045535 0.086820           False
body-parts      138      0.157463 0.065115 0.086820           False

Pooled OLS (descriptive):
spam_distance beta=1815.07, p=0.0169, R^2=0.024

Saved figures:
  spam_fig01_mds_consensus_maps.png
  spam_fig09_rq3_correlation.png
  spam_fig10_cross_task_summary.png
Saved tables:
  table_spam_clusters.csv
  table_spam_rq3_results.csv
  table_spam_consensus_pairs.csv

SpAM notebook complete.
